<a href="https://colab.research.google.com/github/azcsprof/ASU-CSE475-SS25/blob/Unit-5-Lab-2/Unit_5_Lab_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Step 1: Import Required Libraries


In [ ]:
import tensorflow.compat.v1 as tf
tf.disable_eager_execution()
tf.disable_v2_behavior()

from aif360.metrics import ClassificationMetric
from aif360.algorithms.preprocessing.optim_preproc_helpers.data_preproc_functions import load_preproc_data_adult
from aif360.algorithms.inprocessing.adversarial_debiasing import AdversarialDebiasing
from sklearn.preprocessing import MaxAbsScaler
import matplotlib.pyplot as plt

##Step 2: Load and Split the Dataset

In [ ]:
dataset_adult = load_preproc_data_adult()
dataset_adult_train, dataset_adult_test = dataset_adult.split([0.6], shuffle=True)

##Step 3: Define Privileged vs. Unprivileged Groups

In [ ]:
privileged_groups = [{'sex': 1}]   # Male
unprivileged_groups = [{'sex': 0}] # Female

##Step 4: Normalize Features

In [ ]:
min_max_scaler = MaxAbsScaler()
dataset_adult_train.features = min_max_scaler.fit_transform(dataset_adult_train.features)
dataset_adult_test.features = min_max_scaler.transform(dataset_adult_test.features)

##Step 5: Train Adversarial Debiasing Model

In [ ]:
sess = tf.Session()
debiased_model = AdversarialDebiasing(
    privileged_groups=privileged_groups,
    unprivileged_groups=unprivileged_groups,
    scope_name='debiased_classifier',
    debias=True,
    sess=sess
)
debiased_model.fit(dataset_adult_train)

##Step 6: Predict on Test Set

In [ ]:
dataset_adult_pred = debiased_model.predict(dataset_adult_test)

##Step 7: Define Reusable Evaluation Function

In [ ]:
def evaluate_fairness(name, y_true, y_pred):
    m = ClassificationMetric(
        y_true, y_pred,
        unprivileged_groups=unprivileged_groups,
        privileged_groups=privileged_groups
    )
    print(f"=== {name} ===")
    print("Accuracy:", m.accuracy())
    print("Statistical Parity Difference:", m.statistical_parity_difference())
    print("→ A value closer to 0 means both groups receive positive outcomes at similar rates.")
    print("Equal Opportunity Difference:", m.equal_opportunity_difference())
    print("→ A value closer to 0 means both groups have equal true positive rates.")

##Step 8: Evaluate the Debiased Model

In [ ]:
evaluate_fairness("Debiased Model", dataset_adult_test, dataset_adult_pred)

##Step 9: Train Baseline Model (No Debiasing)

In [ ]:
sess_baseline = tf.Session()
baseline_model = AdversarialDebiasing(
    privileged_groups=privileged_groups,
    unprivileged_groups=unprivileged_groups,
    scope_name='baseline_classifier',
    debias=False,
    sess=sess_baseline
)
baseline_model.fit(dataset_adult_train)
baseline_pred = baseline_model.predict(dataset_adult_test)

##Step 10: Evaluate Baseline Model

In [ ]:
evaluate_fairness("Baseline Model", dataset_adult_test, baseline_pred)

##Step 11: Visualize Group Prediction Distributions

In [ ]:
def plot_group_distribution(dataset, title):
    labels = dataset.labels.ravel()
    groups = dataset.protected_attributes.ravel()
    plt.hist([labels[groups == 1], labels[groups == 0]],
             label=["Privileged (Male)", "Unprivileged (Female)"],
             bins=2, align='left', rwidth=0.8)
    plt.xticks([0, 1], ["Negative", "Positive"])
    plt.title(title)
    plt.xlabel("Predicted Label")
    plt.ylabel("Count")
    plt.legend()
    plt.show()

plot_group_distribution(dataset_adult_test, "Actual Label Distribution by Group")
plot_group_distribution(dataset_adult_pred, "Predicted Label Distribution by Group (Debiased)")

##Interactive Widget

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Widget controls
debias_toggle = widgets.ToggleButtons(
    options=[True, False],
    description='Debiasing:',
    value=True
)
epoch_slider = widgets.IntSlider(
    value=10, min=1, max=50, step=1,
    description='Epochs:'
)
adv_weight_slider = widgets.FloatSlider(
    value=0.1, min=0.0, max=1.0, step=0.05,
    description='Adv Weight:'
)

run_button = widgets.Button(description="Train Model")

# Output display
out = widgets.Output()

def on_run_button_clicked(b):
    clear_output(wait=True)
    display(debias_toggle, epoch_slider, adv_weight_slider, run_button, out)
    with out:
        tf.reset_default_graph()
        sess = tf.Session()
        model = AdversarialDebiasing(
            privileged_groups=privileged_groups,
            unprivileged_groups=unprivileged_groups,
            scope_name='interactive_classifier',
            debias=debias_toggle.value,
            sess=sess,
            num_epochs=epoch_slider.value,
            adversary_loss_weight=adv_weight_slider.value
        )
        model.fit(dataset_adult_train)
        pred = model.predict(dataset_adult_test)
        evaluate_fairness("Interactive Model", dataset_adult_test, pred)
        plot_group_distribution(pred, "Prediction Distribution (Interactive)")

run_button.on_click(on_run_button_clicked)

# Display widgets
display(debias_toggle, epoch_slider, adv_weight_slider, run_button, out)